In [1]:
import pandas as pd
from typing import List
import json
import os
os.chdir("../../")

In [2]:
def read_file(fname):
    output = []
    with open(fname) as f:
        for line in f:
            output.append(line.strip())
    return output

def read_jsonl(fname):
    output = []
    with open(fname, 'r', encoding='utf-8') as file:
        for line in file:
            output.append(json.loads(line))
    return output

def read_json(fname):
    with open(fname, 'r', encoding='utf-8') as file:
            output = json.load(file)
    return output

def get_instructions(df, lp, context_type, split="dev"):
    source_lang = lp.split("-")[0]
    target_lang = lp.split("-")[1].replace('pt','pt-br')
    
    xx_yy_instrs = read_file(f"instructions/{context_type}/mt/wmt24_chat_{split.replace('test', 'test_blind')}.{source_lang}-{target_lang}/instructions.txt")
    yy_xx_instrs = read_file(f"instructions/{context_type}/mt/wmt24_chat_{split.replace('test', 'test_blind')}.{target_lang}-{source_lang}/instructions.txt")
    assert len(xx_yy_instrs) + len(yy_xx_instrs) == len(df)
    instructions = []
    j,k=0,0
    
    for i, row in df.iterrows():
        if row["lp"] == f"{source_lang}-{target_lang}":
          instructions.append(xx_yy_instrs[j])
          j+=1
        else:
          # nl-en and it-en are considered here
          instructions.append(yy_xx_instrs[k])
          k+=1
    
    return instructions


def get_pcxmi_log_probs(df, lp, context_type,model_name, split="dev"):
    source_lang = lp.split("-")[0]
    target_lang = lp.split("-")[1].replace('pt','pt-br')

    #read hypothesis logprobs!
    xx_yy_probs_hyp = read_jsonl(f"pcxmi/{context_type}/mt/wmt24_chat_{split.replace('test', 'test_blind')}.{source_lang}-{target_lang}/{model_name}/log_probs_hyp.jsonl")
    xx_yy_probs_ref = read_jsonl(f"pcxmi/{context_type}/mt/wmt24_chat_{split.replace('test', 'test_blind')}.{source_lang}-{target_lang}/{model_name}/log_probs_ref.jsonl")
    yy_xx_probs_hyp = read_jsonl(f"pcxmi/{context_type}/mt/wmt24_chat_{split.replace('test', 'test_blind')}.{target_lang}-{source_lang}/{model_name}/log_probs_hyp.jsonl")
    yy_xx_probs_ref = read_jsonl(f"pcxmi/{context_type}/mt/wmt24_chat_{split.replace('test', 'test_blind')}.{target_lang}-{source_lang}/{model_name}/log_probs_ref.jsonl")
    assert len(xx_yy_probs_hyp) + len(yy_xx_probs_hyp) == len(df)
    assert len(xx_yy_probs_ref) + len(yy_xx_probs_ref) == len(df)
    probs_hyp = []
    probs_ref = []
    j,k=0,0
    for i, row in df.iterrows():
        if row["lp"] == f"{source_lang}-{target_lang}":
          probs_hyp.append(xx_yy_probs_hyp[j]["log_probs"])
          probs_ref.append(xx_yy_probs_ref[j]["log_probs"])
          j+=1
        else:
          # nl-en and it-en are considered here
          probs_hyp.append(yy_xx_probs_hyp[k]["log_probs"])
          probs_ref.append(yy_xx_probs_ref[k]["log_probs"])
          k+=1
    
    return {"log_probs_hyp":probs_hyp,"log_probs_ref":probs_ref}

def get_muda_tags(df, lang, model_name, split="dev"):
    source_lang = lang.split("-")[0]
    target_lang = lang.split("-")[1]
    #read tags!
    xx_yy_hyps_tags = read_json(f"paper_results/{split}_{target_lang}_{model_name}_hyps_tags.json")
    hyps_tags = [inst for conversation in xx_yy_hyps_tags for inst in conversation]
    xx_yy_refs_tags = read_json(f"paper_results/{split}_{target_lang}_{model_name}_refs_tags.json")
    refs_tags = [inst for conversation in xx_yy_refs_tags for inst in conversation]
    if target_lang=="pt":
        assert len(hyps_tags)  == (df.lp==f"en-{target_lang}".replace('pt','pt-br')).sum()
        assert len(refs_tags)  == (df.lp==f"en-{target_lang}".replace('pt','pt-br')).sum()
    else:
        assert len(hyps_tags)  == (df.lp==f"en-{target_lang}").sum()
        assert len(refs_tags)  == (df.lp==f"en-{target_lang}").sum()
    tags_hyp = []
    tags_ref = []
    j=0
    for i, row in df.iterrows():
        if row["lp"] == f"en-{target_lang}":
        # en-xx case:
            tags_hyp.append(hyps_tags[j])
            tags_ref.append(refs_tags[j])
            j+=1
        else:
            # xx-en are considered here
            tags_hyp.append(None)
            tags_ref.append(None)
        
    return {"tags_hyp":tags_hyp,"tags_ref":tags_ref}



In [3]:
LANGS = ["en-de","en-fr","en-ko","en-nl","en-pt"]
SPLITS = ["dev","test"]

In [4]:

keep_columns = ['source_language', 'target_language', 'source', 'reference', 'doc_id',
       'client_id', 'sender', 'tags', 'lp', 'greedy-7b-w-context', 'greedy-7b-w-context-comet',
       'greedy-7b-wo-context', 'greedy-7b-wo-context-comet','greedy-7b-ft-w-context', 'greedy-7b-ft-w-context-comet',
       'greedy-7b-ft-wo-context', 'greedy-7b-ft-wo-context-comet']


data_dict={}
for lang_dir in LANGS:
    data_dict[lang_dir]={}
    for split in SPLITS:
        df = pd.read_csv(f"paper_results/{split}.{lang_dir}.csv")
        df["lp"] = df["source_language"] + "-" + df["target_language"]
        data_dict[lang_dir][split]=df[keep_columns]



In [5]:
ctx_options = {"full_context":"greedy-7b-w-context","full_context_empty_sys":"greedy-7b-ft-w-context","no_context":"greedy-7b-wo-context","no_context_empty_sys":"greedy-7b-ft-wo-context"}
# model_options = ["TowerInstruct-7B-v0.2","TowerInstruct-7B-w-chat-empty-sys","TowerInstruct-7B-v0.2","TowerInstruct-7B-w-chat-empty-sys"]

# mapping_dict = {"full_context_TowerInstruct-7B-v0.2":"greedy-7b-w-context",
#                 "no_context_TowerInstruct-7B-v0.2":"greedy-7b-wo-context",
#                 "full_context_empty_sys_TowerInstruct-7B-w-chat-empty-sys":"greedy-7b-ft-w-context",
#                 "no_context_empty_sys_TowerInstruct-7B-w-chat-empty-sys":"greedy-7b-ft-wo-context",
#                }

instr_dict={}
for lang_dir in LANGS:
    instr_dict[lang_dir]={}
    for split in SPLITS:
        instr_dict[lang_dir][split]={}
        gen_data  = data_dict[lang_dir][split]
        tmp_dict = {}
        for ctx,ctx_val in ctx_options.items():
            instructions = get_instructions(gen_data, lang_dir,context_type=ctx, split=split)
            tmp_dict["instruction-"+ctx_val]=instructions
        instr_df = pd.DataFrame.from_dict(tmp_dict)
        instr_df["source_language"] = gen_data["source_language"]
        instr_df["target_language"] = gen_data["target_language"]
        instr_dict[lang_dir][split] = instr_df
        

In [6]:
data_dict["en-de"]["test"].head(7)

,source_language,target_language,source,reference,doc_id,client_id,sender,tags,lp,greedy-7b-w-context,greedy-7b-w-context-comet,greedy-7b-wo-context,greedy-7b-wo-context-comet,greedy-7b-ft-w-context,greedy-7b-ft-w-context-comet,greedy-7b-ft-wo-context,greedy-7b-ft-wo-context-comet
0,de,en,"NAME-M_TEXT:Einen Moment, ich leite Sie jetzt ...","NAME-M_TEXT:One moment, I will now forward you...",644ce0222d2c2c51df768fd6,0015800001WUWJbAAP,customer,[],de-en,"NAME-M_TEXT:One moment, I will now transfer yo...",0.884888,"NAME-M_TEXT:One moment, I will now transfer yo...",0.884888,"NAME-M_TEXT:One moment, I will now transfer yo...",0.884888,"NAME-M_TEXT:One moment, I will now transfer yo...",0.884888
1,de,en,"Hallo, mein Name ist NAME-M.","Hello, my name is NAME-M.",644ce0222d2c2c51df768fd6,0015800001WUWJbAAP,customer,[],de-en,"Hello, my name is NAME-M.",0.987580,"Hello, my name is NAME-M.",0.987580,"Hello, my name is NAME-M.",0.987580,"Hello, my name is NAME-M.",0.987580
2,de,en,"Einen Moment bitte, ich schaue","One moment please, I am looking",644ce0222d2c2c51df768fd6,0015800001WUWJbAAP,customer,[],de-en,"One moment please, I will look into it.",0.786679,"Just a moment, I'll look it up.",0.690116,"One moment please, I will look it up.",0.837834,"One moment please, I'll look it up.",0.814410
3,de,en,mir jetzt Ihre Unterhaltung mit NAME-M an.,me your conversation with NAME-M now.,644ce0222d2c2c51df768fd6,0015800001WUWJbAAP,customer,[],de-en,I'll now take you off hold with NAME-M.,0.538329,I'm listening to your conversation with NAME-M...,0.756699,I am now listening to your conversation with N...,0.790909,I'm listening to your conversation with NAME-M.,0.771731
4,de,en,"NAME-M_TEXT:Es tut mir leid, anscheinend ist g...","NAME-M_TEXT:I'm sorry, apparently an error has...",644ce0222d2c2c51df768fd6,0015800001WUWJbAAP,customer,[],de-en,"NAME-M_TEXT:I'm sorry, it seems like there was...",0.891846,"NAME-M_TEXT:I am sorry, it seems that an error...",0.911036,"NAME-M_TEXT:I'm sorry, apparently a error has ...",0.930057,"NAME-M_TEXT:I am sorry, it seems that an error...",0.911036
5,en,de,Could you please elaborate on your query so th...,Könnten Sie bitte Ihre Anfrage näher erläutern...,644ce0222d2c2c51df768fd6,0015800001WUWJbAAP,agent,[],en-de,Könnten Sie bitte Ihre Anfrage näher erläutern...,0.962051,"Könnten Sie bitte Ihre Anfrage erläutern, dami...",0.949746,Könnten Sie Ihre Anfrage bitte detaillierter e...,0.959529,Könnten Sie Ihre Anfrage bitte etwas genauer e...,0.959005
6,en,de,Are you still with me?,Besteht die Verbindung noch?,644ce0222d2c2c51df768fd6,0015800001WUWJbAAP,agent,[],en-de,Sind Sie noch dabei?,0.821933,Sind Sie noch bei mir?,0.781136,Sind Sie noch da?,0.858470,Sind Sie noch da?,0.858470


In [7]:
instr_dict["en-de"]["test"].head(7)

,instruction-greedy-7b-w-context,instruction-greedy-7b-ft-w-context,instruction-greedy-7b-wo-context,instruction-greedy-7b-ft-wo-context,source_language,target_language
0,<|im_start|>user\nTranslate the following Germ...,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,<|im_start|>user\nTranslate the following Germ...,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,de,en
1,<|im_start|>user\nContext: NAME-M_TEXT:Einen M...,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,<|im_start|>user\nTranslate the following Germ...,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,de,en
2,<|im_start|>user\nContext: NAME-M_TEXT:Einen M...,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,<|im_start|>user\nTranslate the following Germ...,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,de,en
3,<|im_start|>user\nContext: NAME-M_TEXT:Einen M...,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,<|im_start|>user\nTranslate the following Germ...,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,de,en
4,<|im_start|>user\nContext: NAME-M_TEXT:Einen M...,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,<|im_start|>user\nTranslate the following Germ...,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,de,en
5,<|im_start|>user\nContext: NAME-M_TEXT:Einen M...,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,<|im_start|>user\nTranslate the following Engl...,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,en,de
6,<|im_start|>user\nContext: NAME-M_TEXT:Einen M...,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,<|im_start|>user\nTranslate the following Engl...,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,en,de


In [8]:

ctx_options = ["full_context","full_context_empty_sys","no_context","no_context_empty_sys"]
model_options = ["TowerInstruct-7B-v0.2","TowerInstruct-7B-w-chat-empty-sys","TowerInstruct-7B-v0.2","TowerInstruct-7B-w-chat-empty-sys"]

mapping_dict = {"full_context_TowerInstruct-7B-v0.2":"greedy-7b-w-context",
                "no_context_TowerInstruct-7B-v0.2":"greedy-7b-wo-context",
                "full_context_empty_sys_TowerInstruct-7B-w-chat-empty-sys":"greedy-7b-ft-w-context",
                "no_context_empty_sys_TowerInstruct-7B-w-chat-empty-sys":"greedy-7b-ft-wo-context",
               }
log_probs_dict = {}
for lang_dir in LANGS:
    log_probs_dict[lang_dir]={}
    for split in SPLITS:
        log_probs_dict[lang_dir][split]={}
        gen_data  = data_dict[lang_dir][split]
        tmp_dict = {}
        for ctx,model in zip(ctx_options,model_options):
            log_probs = get_pcxmi_log_probs(gen_data, lang_dir,context_type=ctx,model_name=model, split=split)
            tmp_dict[mapping_dict[ctx+"_"+model]+"_log_probs_hyp"] = log_probs["log_probs_hyp"]
            tmp_dict[mapping_dict[ctx+"_"+model]+"_log_probs_ref"] = log_probs["log_probs_ref"]            
        log_probs_df = pd.DataFrame.from_dict(tmp_dict)
        log_probs_df["source_language"] = gen_data["source_language"]
        log_probs_df["target_language"] = gen_data["target_language"]
        log_probs_dict[lang_dir][split] = log_probs_df
        

In [9]:
log_probs_dict["en-de"]["dev"].head(3)

,greedy-7b-w-context_log_probs_hyp,greedy-7b-w-context_log_probs_ref,greedy-7b-ft-w-context_log_probs_hyp,greedy-7b-ft-w-context_log_probs_ref,greedy-7b-wo-context_log_probs_hyp,greedy-7b-wo-context_log_probs_ref,greedy-7b-ft-wo-context_log_probs_hyp,greedy-7b-ft-wo-context_log_probs_ref,source_language,target_language
0,"[-0.15230952203273773, -0.21945787966251373, -...","[-0.15279555320739746, -2.54947829246521, -2.4...","[-0.07075800746679306, -0.27593278884887695, -...","[-7.75825834274292, -2.904130220413208, -2.529...","[-0.1549936830997467, -0.2183438390493393, -0....","[-0.15279555320739746, -2.54947829246521, -2.4...","[-0.07075779139995575, -0.27593278884887695, -...","[-7.75825834274292, -2.904130220413208, -2.529...",de,en
1,"[-0.32573428750038147, -0.10851939767599106, -...","[-0.32651644945144653, -3.062605619430542, -1....","[-0.23615194857120514, -0.16074395179748535, -...","[-13.173651695251465, -2.217759370803833, -1.1...","[-0.3578929603099823, -0.18080858886241913, -0...","[-0.3579587936401367, -2.7138214111328125, -1....","[-0.3798878788948059, -0.4673590064048767, -0....","[-15.036137580871582, -2.36857008934021, -1.58...",de,en
2,"[-0.05889945104718208, -0.01523324754089117, -...","[-0.05937430262565613, -0.015315194614231586, ...","[-0.006852341815829277, -0.0012232924345880747...","[-11.756852149963379, -0.002458765869960189, -...","[-0.038545310497283936, -0.021481232717633247,...","[-0.038521911948919296, -0.02155403420329094, ...","[-0.03168584406375885, -0.00713846180588007, -...","[-14.406685829162598, -0.010023378767073154, -...",de,en


In [10]:
load_options = ["greedy-7b-w-context","greedy-7b-wo-context","greedy-7b-ft-w-context","greedy-7b-ft-wo-context"]

muda_tags_dict = {}
for lang_dir in LANGS:
    muda_tags_dict[lang_dir]={}
    for split in SPLITS:
        muda_tags_dict[lang_dir][split]={}
        gen_data  = data_dict[lang_dir][split]
        tmp_dict = {}
        for model in load_options:
            muda_tags = get_muda_tags(gen_data, lang_dir, model, split=split)
            tmp_dict[model+"_tags_hyp"] = muda_tags["tags_hyp"]
            tmp_dict[model+"_tags_ref"] = muda_tags["tags_ref"]            
        tags_df = pd.DataFrame.from_dict(tmp_dict)
        tags_df["source_language"] = gen_data["source_language"]
        tags_df["target_language"] = gen_data["target_language"]
        muda_tags_dict[lang_dir][split] = tags_df
        

In [11]:
muda_tags_dict["en-de"]["test"].head(8)

,greedy-7b-w-context_tags_hyp,greedy-7b-w-context_tags_ref,greedy-7b-wo-context_tags_hyp,greedy-7b-wo-context_tags_ref,greedy-7b-ft-w-context_tags_hyp,greedy-7b-ft-w-context_tags_ref,greedy-7b-ft-wo-context_tags_hyp,greedy-7b-ft-wo-context_tags_ref,source_language,target_language
0,None,None,None,None,None,None,None,None,de,en
1,None,None,None,None,None,None,None,None,de,en
2,None,None,None,None,None,None,None,None,de,en
3,None,None,None,None,None,None,None,None,de,en
4,None,None,None,None,None,None,None,None,de,en
5,"[{'token': 'Könnten', 'tags': []}, {'token': '...","[{'token': 'Könnten', 'tags': []}, {'token': '...","[{'token': 'Könnten', 'tags': []}, {'token': '...","[{'token': 'Könnten', 'tags': []}, {'token': '...","[{'token': 'Könnten', 'tags': []}, {'token': '...","[{'token': 'Könnten', 'tags': []}, {'token': '...","[{'token': 'Könnten', 'tags': []}, {'token': '...","[{'token': 'Könnten', 'tags': []}, {'token': '...",en,de
6,"[{'token': 'Besteht', 'tags': []}, {'token': '...","[{'token': 'Besteht', 'tags': []}, {'token': '...","[{'token': 'Besteht', 'tags': []}, {'token': '...","[{'token': 'Besteht', 'tags': []}, {'token': '...","[{'token': 'Besteht', 'tags': []}, {'token': '...","[{'token': 'Besteht', 'tags': []}, {'token': '...","[{'token': 'Besteht', 'tags': []}, {'token': '...","[{'token': 'Besteht', 'tags': []}, {'token': '...",en,de
7,None,None,None,None,None,None,None,None,de,en


check if sentence-level comet scores with context are greater than without. Set true if they are. 

In [12]:
load_options = [["greedy-7b-w-context","greedy-7b-wo-context"],["greedy-7b-ft-w-context","greedy-7b-ft-wo-context"]]

for lang_dir in LANGS:
    for split in SPLITS:
        for model_w_ctx,model_wo_ctx in load_options:
            df = data_dict[lang_dir][split].copy()
            comet_greater_df = data_dict[lang_dir][split][model_w_ctx+"-comet"]>data_dict[lang_dir][split][model_wo_ctx+"-comet"]
            df.loc[:, 'comet-greater'] = comet_greater_df
            data_dict[lang_dir][split] = df


In [13]:
data_dict["en-de"]["test"].head(7)

,source_language,target_language,source,reference,doc_id,client_id,sender,tags,lp,greedy-7b-w-context,greedy-7b-w-context-comet,greedy-7b-wo-context,greedy-7b-wo-context-comet,greedy-7b-ft-w-context,greedy-7b-ft-w-context-comet,greedy-7b-ft-wo-context,greedy-7b-ft-wo-context-comet,comet-greater
0,de,en,"NAME-M_TEXT:Einen Moment, ich leite Sie jetzt ...","NAME-M_TEXT:One moment, I will now forward you...",644ce0222d2c2c51df768fd6,0015800001WUWJbAAP,customer,[],de-en,"NAME-M_TEXT:One moment, I will now transfer yo...",0.884888,"NAME-M_TEXT:One moment, I will now transfer yo...",0.884888,"NAME-M_TEXT:One moment, I will now transfer yo...",0.884888,"NAME-M_TEXT:One moment, I will now transfer yo...",0.884888,False
1,de,en,"Hallo, mein Name ist NAME-M.","Hello, my name is NAME-M.",644ce0222d2c2c51df768fd6,0015800001WUWJbAAP,customer,[],de-en,"Hello, my name is NAME-M.",0.987580,"Hello, my name is NAME-M.",0.987580,"Hello, my name is NAME-M.",0.987580,"Hello, my name is NAME-M.",0.987580,False
2,de,en,"Einen Moment bitte, ich schaue","One moment please, I am looking",644ce0222d2c2c51df768fd6,0015800001WUWJbAAP,customer,[],de-en,"One moment please, I will look into it.",0.786679,"Just a moment, I'll look it up.",0.690116,"One moment please, I will look it up.",0.837834,"One moment please, I'll look it up.",0.814410,True
3,de,en,mir jetzt Ihre Unterhaltung mit NAME-M an.,me your conversation with NAME-M now.,644ce0222d2c2c51df768fd6,0015800001WUWJbAAP,customer,[],de-en,I'll now take you off hold with NAME-M.,0.538329,I'm listening to your conversation with NAME-M...,0.756699,I am now listening to your conversation with N...,0.790909,I'm listening to your conversation with NAME-M.,0.771731,True
4,de,en,"NAME-M_TEXT:Es tut mir leid, anscheinend ist g...","NAME-M_TEXT:I'm sorry, apparently an error has...",644ce0222d2c2c51df768fd6,0015800001WUWJbAAP,customer,[],de-en,"NAME-M_TEXT:I'm sorry, it seems like there was...",0.891846,"NAME-M_TEXT:I am sorry, it seems that an error...",0.911036,"NAME-M_TEXT:I'm sorry, apparently a error has ...",0.930057,"NAME-M_TEXT:I am sorry, it seems that an error...",0.911036,True
5,en,de,Could you please elaborate on your query so th...,Könnten Sie bitte Ihre Anfrage näher erläutern...,644ce0222d2c2c51df768fd6,0015800001WUWJbAAP,agent,[],en-de,Könnten Sie bitte Ihre Anfrage näher erläutern...,0.962051,"Könnten Sie bitte Ihre Anfrage erläutern, dami...",0.949746,Könnten Sie Ihre Anfrage bitte detaillierter e...,0.959529,Könnten Sie Ihre Anfrage bitte etwas genauer e...,0.959005,True
6,en,de,Are you still with me?,Besteht die Verbindung noch?,644ce0222d2c2c51df768fd6,0015800001WUWJbAAP,agent,[],en-de,Sind Sie noch dabei?,0.821933,Sind Sie noch bei mir?,0.781136,Sind Sie noch da?,0.858470,Sind Sie noch da?,0.858470,False


In [14]:
# concat_df = pd.concat((data_dict["de"]["test"],instr_dict["de"]["test"],log_probs_dict["de"]["test"],muda_tags_dict["de"]["test"]),axis=1)

# final_df = concat_df.loc[:,~concat_df.columns.duplicated()].copy()

# print(final_df.columns)


##### Now we merge all the dfs for each (lang,split) key.

In [14]:
merged_dict ={}
for lang_dir in LANGS:
    merged_dict[lang_dir]={}
    for split in SPLITS:
        concat_df = pd.concat((data_dict[lang_dir][split],instr_dict[lang_dir][split],log_probs_dict[lang_dir][split],muda_tags_dict[lang_dir][split]),axis=1)
        final_df = concat_df.loc[:,~concat_df.columns.duplicated()].copy()
        merged_dict[lang_dir][split] = final_df

# now we split the data into all directions
save_dict ={}
for lang_dir in LANGS:
    source_lang = lang_dir.split("-")[0]
    target_lang = lang_dir.split("-")[1]
    save_dict[source_lang+"-"+target_lang]={}
    save_dict[target_lang+"-"+source_lang]={}
    for split in SPLITS:
        xx_yy_df = merged_dict[lang_dir][split].loc[(merged_dict[lang_dir][split].source_language==source_lang)]
        yy_xx_df = merged_dict[lang_dir][split].loc[(merged_dict[lang_dir][split].source_language==target_lang)]
        save_dict[source_lang+"-"+target_lang][split] = xx_yy_df.reset_index().copy()
        save_dict[target_lang+"-"+source_lang][split] = yy_xx_df.reset_index().copy()

##### SOS: 
The index column is kept in case we want to know the correct "timestep" of the instance (turn) in the conversation. To do this we should merge the dataframes (e.g. en-de merge with de-en) and sort by the index. In this case the instances will be again in the correct order.

Now we save all the data in json format as dicts:

In [15]:
# save_dict["de-en"]["test"].head(3).to_dict(orient='records')

In [16]:

for lang_dir in save_dict.keys():
    for split in save_dict[lang_dir].keys():
        fname = f'./alti_prep_data/full_context/{split}_{lang_dir}_datadf.json'
        data = save_dict[lang_dir][split].to_dict(orient='records')
        with open(fname, 'w') as f:
            json.dump(data, f)